# F-OAL: CIFAR-100 Full Reproduction (NeurIPS 2024)
## Class Incremental Learning with Forward-only Online Analytic Learning

This notebook provides the end-to-end execution pipeline for replicating the official CIFAR-100 benchmark from the paper:
> **F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning**  
> *Yuchen Liu, et al. NeurIPS 2024*

### Benchmark Targets (CIFAR-100 B0-10tasks, Table 2 of Paper)
- **Average Incremental Accuracy ($A_{\text{avg}}$):** ~91.1%
- **Last Task Accuracy ($A_{\text{last}}$):** ~86.5%
- **Average Forgetting ($F$):** ~5.5%

### Experimental Protocol
- **Backbone:** Frozen ViT-B/16 (`vit_base_patch16_224.augreg_in1k`)
- **Feature Fusion:** Multi-layer CLS token averaging (layers 0-11) + final LayerNorm
- **Expansion:** Smooth Random Projection to $D = 1000$, $\gamma = 1.0$
- **Learner:** Online Recursive Least Squares (Woodbury update, $\lambda = 0.1$, float64)
- **Data Protocol:** 10 tasks $\times$ 10 classes, streaming batch size 10, strictly **one-pass** (single epoch)
- **Sample Limits:** None (full 500 train / 100 test per class)

## 1. Hardware & Environment Verification
Ensure that an NVIDIA GPU (T4, P100, V100, A100, or L4) is active in this runtime.

In [ ]:
!nvidia-smi

In [ ]:
import torch
import sys

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    raise SystemError("CUDA GPU not detected! Please enable GPU accelerator in Kaggle/Colab runtime settings.")

## 2. Clone Repository & Install Dependencies
If running on Kaggle or Google Colab, clone the repository and install required packages.

In [ ]:
# If running outside the repository root, uncomment and clone:
# !git clone https://github.com/caoban123/F-OAL.git
# %cd F-OAL

!pip install -q -r requirements.txt

## 3. Run Preflight Test Suite
Verify mathematical correctness, Woodbury RLS equivalence, layer normalization, and run completeness validation.

In [ ]:
!python -m pytest -v

## 4. Single-Seed Full Reproduction (Seed 42)
Run F-OAL on the full CIFAR-100 dataset across all 10 tasks with online streaming batch size 10.

*(Optional: Pass your W&B API Key via `WANDB_API_KEY` or run offline with `--wandb-offline`)*

In [ ]:
# Set W&B mode: offline or online
# import os; os.environ["WANDB_API_KEY"] = "your_api_key_here"

!python -m scripts.train \
    --config configs/cifar100.yaml \
    --seed 42 \
    --device cuda \
    --run-name foal_full_cifar100_seed42 \
    --wandb \
    --wandb-offline

## 5. Independent Checkpoint Evaluation
Verify evaluation reproducibility by reloading the final model checkpoint (`task_10.pt`) and evaluating against all 10 tasks.

In [ ]:
!python -m scripts.evaluate \
    --checkpoint results/foal_full_cifar100_seed42/task_10.pt \
    --device cuda

## 6. Official Results Table & Summary
Summarize the reproduction run using `scripts/summarize_runs.py`.
The summarizer strictly isolates protocols, prevents pilot-data mixing, and verifies full checkpoint completeness.

In [ ]:
!python -m scripts.summarize_runs --results-dir results

## 7. Multi-Seed Full Reproduction (Seeds 42, 123, 2026)
To produce statistically solid benchmark figures (Mean ± Std), run across 3 seeds using `scripts/run_seeds.py`.
`--skip-completed` ensures that if seed 42 is already complete with checkpoints, it will not be recomputed.

In [ ]:
!python -m scripts.run_seeds \
    --seeds 42 123 2026 \
    --config configs/cifar100.yaml \
    --device cuda \
    --run-name-prefix foal_full_3seeds \
    --wandb \
    --wandb-offline \
    --skip-completed

In [ ]:
# Generate official multi-seed aggregation summary
!python -m scripts.summarize_runs --results-dir results

## 8. Export & Archive Results
Compress all result logs, metrics, CSVs, and metadata into a zip archive for downloading.

In [ ]:
!zip -r foal_cifar100_results.zip results/